# Step 3: real training on Overture
Overture train → val only; no test split here (the single test run comes after the pooling × seeds runs are compared). `evaluate`, `is_better` and `broken_run_problems` live in `13_real_training.py` (moved there 2026-10-23, so `14_training_analysis.ipynb` can import them); each one's unit tests stay here, under its section. `run_training` stays in this notebook (it reads the loaders built in §2).

**Spec (decided 2026-10-21)**

| Row | Decision |
|---|---|
| Data | Overture only (train → val); Stamford dropped from Step 3; case2 left as is (172 train rows, no weighting) |
| Model | `SiameseMatcher(Encoder(...))` from `08_encoder_step.py`: BiLSTM 1 × 128, emb 32, max length 85; pooling mean or max |
| Optimiser | Adam lr 1e-3, clip 1.0, batch 64, `BCEWithLogitsLoss` (no `pos_weight`) |
| Threshold | 0.5 (same as 1b) |
| Watched each epoch | train loss, val loss, val non-identical found / case1 / easy |
| Best epoch | highest val non-identical found **among epochs with val case1 ≥ 98%**; tie → lower val loss |
| Stop | 3 epochs without a new best (`PATIENCE`), or 20 epochs (`MAX_EPOCHS`) |
| No epoch qualifies | keep the last epoch's weights, `best = None` (patience then ends the run after 3 epochs) |
| Runs | mean vs max × seeds 0, 1, 2 = 6 runs (§8); "better" judged on mean ± spread over seeds |
| Broken-run checks | untrained val loss ≈ 0.693; train loss parked at ≈ 0.325 = learned only the base rate; no NaN; val found > 0% |
| Saved per run | best `state_dict` + vocab + config + seed + best row → `checkpoints/overture_siamese_seed_{seed}_pooling_{pool}.pt`; history → `checkpoints/history_seed_{seed}_pooling_{pool}.csv` |
| Reference (val) | LogReg 1b: found 81.8 · RF: found 85.4 · case1 99.5–99.8 · easy 100.0. Prediction (David): easy < 98% (no ZIP) |

| § | What | Where |
|---|---|---|
| 1 | Truncation count | 6 train / 0 val strings > 85 chars (Diagnostics) |
| 2 | Datasets + loaders | `train_loader` (shuffled), `val_loader` (not shuffled) |
| 3 | `evaluate` (+ test) | val loss + found / case1 / easy |
| 4 | Timing | ≈ 4.1 min/epoch (Diagnostics) |
| 5 | `is_better` (+ tests) | early-stopping rule |
| 6 | `run_training` | one run with patience, returns history / best / weights |
| 7 | `broken_run_problems` (+ tests) | four checks on a run's history; [] = healthy |
| 8 | Experiment queue + summary | 6 runs, each saved when it finishes, finished runs skipped on re-run; summary open |

In [1]:
%load_ext autoreload
%autoreload 2

import copy
import os
import sys
import time
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd
import numpy as np
import torch
from torch import nn
import itertools
from torch.utils.data import DataLoader, Subset

from helpers.pairs import load_pairs
from helpers.check_model_performance import review_typical_slices
import math

build_vocab = import_module("siamese_model.01_vocab_step").build_vocab          # digit-prefixed filenames -> can't use `from ... import`
_model_step = import_module("siamese_model.08_encoder_step")
Encoder, SiameseMatcher = _model_step.Encoder, _model_step.SiameseMatcher
_data_step = import_module("siamese_model.11_data_plumbing")
PairDataset, make_collate = _data_step.PairDataset, _data_step.make_collate
_train_step = import_module("siamese_model.12_training_step")
train_epoch, predict = _train_step.train_epoch, _train_step.predict
_real_step = import_module("siamese_model.13_real_training")
evaluate, is_better, broken_run_problems = _real_step.evaluate, _real_step.is_better, _real_step.broken_run_problems
load_run, rebuild_model = _real_step.load_run, _real_step.rebuild_model
THRESHOLD, CASE1_GUARDRAIL, CHECKPOINT_DIR = _real_step.THRESHOLD, _real_step.CASE1_GUARDRAIL, _real_step.CHECKPOINT_DIR

pd.set_option("display.max_rows", None)        # all rows
pd.set_option("display.max_columns", None)     # all columns
pd.set_option("display.max_colwidth", None)    # full text inside each cell (long address strings)
pd.set_option("display.width", None)           # don't wrap/clip the table at a fixed character width

In [2]:
SEED = 0
BATCH_SIZE = 64
LR = 1e-3
N_TIMING_BATCHES = 200      # §4: time this many batches, extrapolate to a full epoch
MAX_EPOCHS = 20           # §4: ~4.1 min/epoch -> worst case ~83 min (Diagnostics)
PATIENCE = 3
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
# THRESHOLD, CASE1_GUARDRAIL, CHECKPOINT_DIR: from 13_real_training.py (setup cell)


In [3]:
ot_train, ot_train_label = load_pairs("overture", "train")
ot_val, ot_val_label = load_pairs("overture", "val")
vocab = build_vocab(ot_train)          # train split only
print(len(ot_train), len(ot_val), len(vocab))

202568 41865 40


## 1. Truncation count
Strings longer than 85 characters before `normalize` cuts them, per column and split. Result in Diagnostics.

In [4]:
print(len(ot_train[ot_train["address_a"].str.len() > 85]))
print(len(ot_train[ot_train["address_b"].str.len() > 85]))
print(len(ot_val[ot_val["address_a"].str.len() > 85]))
print(len(ot_val[ot_val["address_b"].str.len() > 85]))

# print(ot_train[ot_train["address_a"].str.len() > 85]["address_a"])
print(ot_train[ot_train["address_b"].str.len() > 85][["address_b", "neg_type"]])

6
6
0
0
                                                                                                                        address_b  \
18026   WHETTEN GRADUATE CENTER SECOND FLOOR UNIVERSITY OF CONNECTICUT 438 WHITNEY RD EXTENSION UNIT-1152 STORRS 06269-1152 UNITE   
130983  WHETTEN GRADUATE CENTER SECOND FLOOR UNIVERSITY OF CONNECTICUT 438 WHITNEY RD EXTENSION UNIT-1152 STORRS 06269-1152 UNITE   
162764  WHETTEN GRADUATE CENTER SECOND FLOOR UNIVERSITY OF CONNECTICUT 438 WHITNEY RD EXTENSION UNIT-1152 STORRS 06269-1152 UNITE   
195747  WHETTEN GRADUATE CENTER SECOND FLOOR UNIVERSITY OF CONNECTICUT 438 WHITNEY RD EXTENSION UNIT-1152 STORRS 06269-1152 UNITE   
198904  WHETTEN GRADUATE CENTER SECOND FLOOR UNIVERSITY OF CONNECTICUT 438 WHITNEY RD EXTENSION UNIT-1152 STORRS 06269-1152 UNITE   
238879  WHETTEN GRADUATE CENTER SECOND FLOOR UNIVERSITY OF CONNECTICUT 438 WHITNEY RD EXTENSION UNIT-1152 STORRS 06269-1152 UNITE   

       neg_type  
18026     case0  
130983    case3  
162764

## 2. Datasets and loaders
Train shuffled; val **not** shuffled (`predict` returns probabilities in loader order). `run_training` reads these global loaders, so no other cell may rebuild them.

In [5]:
train_dataset = PairDataset(ot_train, ot_train_label)
val_dataset = PairDataset(ot_val, ot_val_label)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle = True, collate_fn = make_collate(vocab))
val_loader = DataLoader(val_dataset, batch_size=64, shuffle = False, collate_fn= make_collate(vocab))

## 3. `evaluate` (in `13_real_training.py`)
One model's val numbers as a dict: `{"val_loss", "found", "case1", "easy"}` (% correct on those slices, same slices as `results.md`). The last lines are the untrained-model check (Diagnostics).

In [6]:
# evaluate lives in 13_real_training.py; the untrained-model check (Diagnostics) stays here
evaluation_model = SiameseMatcher(Encoder(len(vocab)))
untrained = evaluate(evaluation_model, val_loader, ot_val, ot_val_label)
print(untrained)

{'val_loss': 0.7050352692604065, 'found': 100.0, 'case1': 0.036281837312241494, 'easy': 0.19276704521644386}


In [7]:
# Unit test for evaluate, on the untrained model's result from the cell above (no extra val pass)
assert set(untrained) == {"val_loss", "found", "case1", "easy"}
assert all(0 <= untrained[k] <= 100 for k in ["found", "case1", "easy"])
assert math.isfinite(untrained["val_loss"])
assert all(type(v) is float for v in untrained.values())
assert abs(untrained["val_loss"] - 0.693) < 0.05

## 4. Timing
Times `N_TIMING_BATCHES` training batches (extrapolated to a full epoch) and one val pass, on a throwaway model. Result and the `MAX_EPOCHS` / `PATIENCE` decision in Diagnostics.

In [8]:
torch.manual_seed(SEED)
timing_model = SiameseMatcher(Encoder(len(vocab)))
timing_opt = torch.optim.Adam(timing_model.parameters(), lr=LR)
subset_loader = DataLoader(Subset(train_dataset, range(N_TIMING_BATCHES * BATCH_SIZE)), batch_size=BATCH_SIZE, collate_fn = make_collate(vocab))

# train/val loaders come from §2 (train shuffled); don't rebuild them here

t0 = time.perf_counter()
train_epoch(timing_model, subset_loader, timing_opt, nn.BCEWithLogitsLoss())
train_seconds = time.perf_counter() - t0
print(f"train: {train_seconds / N_TIMING_BATCHES:.3f} s/batch -> {train_seconds / N_TIMING_BATCHES * len(train_loader) / 60:.1f} min/epoch")

t0 = time.perf_counter()
predict(timing_model, val_loader)
print(f"val pass: {time.perf_counter() - t0:.1f} s")

train: 0.067 s/batch -> 3.5 min/epoch
val pass: 11.0 s


## 5. `is_better(row, best)` (in `13_real_training.py`)
The early-stopping rule: a row below `CASE1_GUARDRAIL` never qualifies; the first qualifying row is best; then higher `found` wins, a tie goes to the lower `val_loss`.

In [10]:
# Unit tests for is_better (hand-made rows; Claude wrote these at David's request, 2026-10-22)
r = lambda found, case1=99.0, val_loss=0.1: {"found": found, "case1": case1, "val_loss": val_loss}

assert is_better(r(100, case1=0), None) is False         # below guardrail, nothing yet (the untrained model's profile)
assert is_better(r(50), None) is True                    # first qualifying row
assert is_better(r(60), r(50)) is True                   # higher found
assert is_better(r(90, case1=97), r(50)) is False        # higher found but below guardrail
assert is_better(r(50, val_loss=0.1), r(50, val_loss=0.2)) is True    # tie on found -> lower val loss
assert is_better(r(50, val_loss=0.2), r(50, val_loss=0.1)) is False   # tie on found, higher val loss
assert is_better(r(40), r(50)) is False                  # lower found
assert is_better(r(50, case1=98.0), None) is True        # exactly at the guardrail counts
print("is_better: all tests pass")

is_better: all tests pass


## 6. `run_training`
One full run: seed → model → epochs of `train_epoch` + `evaluate`; keeps a deep copy of the best epoch's weights; stops after `PATIENCE` epochs without a new best. If no epoch qualifies, returns the last epoch's weights with `best = None`.

In [11]:
def run_training(seed=SEED, pooling="mean"):
    torch.manual_seed(seed)
    model = SiameseMatcher(Encoder(len(vocab),pooling = pooling))
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr = LR)
    loss_fn = nn.BCEWithLogitsLoss()

    best, best_state, epochs_since_best = None, None, 0
    history = []
    for epoch in range(MAX_EPOCHS):
        train_loss = train_epoch(model, train_loader, opt, loss_fn)

        row = {"epoch": epoch, "train_loss": train_loss, **evaluate(model, val_loader, ot_val, ot_val_label)}
        print(row)
        history.append(row)

        if is_better(row, best):
            best, best_state, epochs_since_best = row, copy.deepcopy(model.state_dict()), 0
        else:
            epochs_since_best += 1
            if epochs_since_best >= PATIENCE:
                break
    if best is None:                                      # no epoch reached the case1 guardrail -> keep the last epoch's weights
        best_state = copy.deepcopy(model.state_dict())
    return pd.DataFrame(history), best, best_state

## 7. Broken-run checks (`broken_run_problems` in `13_real_training.py`)
A function `broken_run_problems(history)` → list of problems (empty = healthy), called on each run's history by the queue (§8), so it is defined first:

| Check | Rule |
|---|---|
| No NaN | `history.isna().any().any()` is False |
| Beats a coin flip after 1 epoch | `history["val_loss"].iloc[0] < 0.693` |
| Not stuck at the base rate | `history["train_loss"].min() < 0.30` |
| Finds something | `history["found"].max() > 0` |

Test it on the broken run's history (2026-10-21, Diagnostics): it must return at least one problem.

In [12]:
# broken_run_problems lives in 13_real_training.py (Claude wrote it at David's request, 2026-10-22); tests stay here
# Tests: a known-broken run must be flagged, a healthy one must not
broken = pd.read_csv("broken/history_mean_seed0.csv")
assert broken_run_problems(broken) != []
print("broken run:", broken_run_problems(broken))          # expected: only the epoch-0 val loss check fires

healthy = pd.DataFrame({"train_loss": [0.05, 0.02], "val_loss": [0.03, 0.02], "found": [78.0, 80.0]})
assert broken_run_problems(healthy) == []

assert broken_run_problems(healthy.assign(val_loss=[float("nan"), 0.02])) != []       # NaN
assert broken_run_problems(healthy.assign(train_loss=[0.33, 0.32])) != []             # stuck at the base rate
assert broken_run_problems(healthy.assign(found=[0.0, 0.0])) != []                    # finds nothing
print("broken_run_problems: all tests pass")


broken run: ['val loss after epoch 0 = 0.992 (coin flip = 0.693)']
broken_run_problems: all tests pass


## 8. Experiment queue: pooling × seeds
6 runs back to back (worst case ≈ 8.3 h). Each run is saved as soon as it finishes; re-running the cell skips runs already saved and carries on.

**To start:** restart the kernel, run every cell from the top through §7, keep the Mac awake (`caffeinate -dims` in a terminal), run this cell, and watch epoch 0 of `mean` / seed 0: expect val loss well below 0.693 and found > 0 (the first shuffled run had found ≈ 79, val loss ≈ 0.02).

In [ ]:
for pool_mode, seed in itertools.product(["mean", "max"], [0,1,2]):
    if os.path.exists(f"checkpoints/overture_siamese_seed_{seed}_pooling_{pool_mode}.pt"):
        continue

    history, best, best_state = run_training(seed = seed, pooling = pool_mode)
    
    history.to_csv(
    os.path.join(CHECKPOINT_DIR, f"history_seed_{seed}_pooling_{pool_mode}.csv"), index = False
    )
    torch.save({
        "state_dict": best_state, "vocab": vocab, 
        "config": {"emb_dim": 32, "hidden": 128, "pooling": pool_mode, "max_len": 85},
        "seed": seed, "best":best}, f"checkpoints/overture_siamese_seed_{seed}_pooling_{pool_mode}.pt")
    print(f"finishing seed:{seed} | pool:{pool_mode} -> best: {best}")
    print(f"problems: {broken_run_problems(history)}")

{'epoch': 0, 'train_loss': 0.04726416479196941, 'val_loss': 0.024339793249964714, 'found': 78.67001254705144, 'case1': 99.84761628328859, 'easy': 99.94133176884716}
{'epoch': 1, 'train_loss': 0.018179889862879092, 'val_loss': 0.017263395711779594, 'found': 81.55583437892095, 'case1': 99.73877077135187, 'easy': 99.95390353266563}
{'epoch': 2, 'train_loss': 0.013840107476110116, 'val_loss': 0.01505804993212223, 'found': 82.55959849435382, 'case1': 99.82584718090123, 'easy': 99.96647529648409}
{'epoch': 3, 'train_loss': 0.01153011055970613, 'val_loss': 0.015324529260396957, 'found': 82.1831869510665, 'case1': 99.85487265075103, 'easy': 99.97485647236307}
{'epoch': 4, 'train_loss': 0.009809347801747232, 'val_loss': 0.01494328398257494, 'found': 84.81806775407779, 'case1': 99.5646179522531, 'easy': 99.96228470854462}
{'epoch': 5, 'train_loss': 0.008785670704643173, 'val_loss': 0.015319195576012135, 'found': 83.5633626097867, 'case1': 99.7315144038894, 'easy': 99.94133176884716}
{'epoch': 6,

**Summary (open, after the queue):** one row per run from `torch.load(path)["best"]` → DataFrame; mean ± spread of `found` per pooling. Also reload one checkpoint from its own `config` + `vocab`, `evaluate` it, and check it matches `best` on `val_loss`, `found`, `case1`, `easy` (proves a file alone rebuilds the model).

In [14]:
rows = []
for pool_mode, seed in itertools.product(["mean", "max"], [0,1,2]):
    history, ckpt = load_run(seed, pool_mode)

    rows.append({"pooling": pool_mode, "seed": seed, **(ckpt["best"] or {}), "epochs_run": len(history), "problems": broken_run_problems(history)})

    # reload check: the file alone rebuilds the model that produced `best` (skipped when no epoch qualified)
    if ckpt["best"] is None:
        continue
    reloaded = rebuild_model(ckpt)
    after = evaluate(reloaded, val_loader, ot_val, ot_val_label)
    for k in ["val_loss", "found", "case1", "easy"]:
        assert after[k] == ckpt["best"][k], (k, after[k], ckpt["best"][k])

summary = pd.DataFrame(rows)
display(summary)
summary.groupby("pooling")[["found", "case1", "easy"]].agg(["mean", "min", "max"])


,pooling,seed,epoch,train_loss,val_loss,found,case1,easy,epochs_run,problems
0,mean,0,10,0.003717,0.016915,87.076537,99.434003,99.932951,14,[]
1,mean,1,4,0.010259,0.014307,87.076537,99.499311,99.899426,8,[]
2,mean,2,5,0.008865,0.015387,90.840652,99.201800,99.878473,9,[]
3,max,0,9,0.004218,0.018681,88.958595,99.274363,99.895235,13,[]
4,max,1,6,0.007373,0.015520,88.205772,99.317901,99.886854,10,[]
5,max,2,5,0.008196,0.018080,90.213300,98.955083,99.832376,9,[]


found                            case1                        \
              mean        min        max       mean        min        max   
pooling                                                                     
max      89.125889  88.205772  90.213300  99.182449  98.955083  99.317901   
mean     88.331242  87.076537  90.840652  99.378371  99.201800  99.499311   

              easy                        
              mean        min        max  
pooling                                   
max      99.871489  99.832376  99.895235  
mean     99.903616  99.878473  99.932951

## Diagnostics
One-off queries and their results (question → result → query). Kept out of the pipeline cells.

- *Truncation (§1), 2026-10-21:* how many strings does the 85-character cut affect? → **6 in train (`address_b`), 0 in val**, all the same Foursquare string (`WHETTEN GRADUATE CENTER … STORRS 06269-1152 UNITE…`; 1 positive + 5 easy negatives). Negligible. The cut-off tail is town + ZIP, i.e. some address strings do contain ZIP/city text (relevant to the open ZIP-text question). Measured on raw lengths (no whitespace collapse; the strings are already cleaned). Query: §1 cell.
- *Timing (§4), 2026-10-21, CPU, batch 64:* how long does a run take, worst case? → train ≈ **204.3 s/epoch** (200 batches timed, × 3,166 / 200 ≈ 0.065 s/batch) + val pass ≈ **43.6 s** ⇒ **≈ 248 s ≈ 4.1 min per epoch**. Worst case = MAX_EPOCHS × 248 s: 15 → ≈ 62 min · **20 → ≈ 83 min** · 50 → ≈ 3.4 h; the follow-up (mean/max × 3 seeds = 6 runs) × 6 (20 → ≈ 8.3 h). Decision: **MAX_EPOCHS = 20, PATIENCE = 3**; CPU is enough, no Colab. Query: §4 cell, separate `perf_counter` timers around `train_epoch` (subset) and `predict(…, val_loader)`.
- *Untrained model on val (§3), 2026-10-21:* → val loss **0.705** (≈ 0.693 ✓), found 100%, case1 0.01%, easy 0.02%: before training every output is ≥ 0.5, so it calls everything a match. Query: last lines of the §3 cell.
- *Broken run, 2026-10-21 (mean, seed 0, 20 epochs):* train loss 0.036 → 0.0008 but val loss **0.99** at epoch 0 (worse than a coin flip), found 0% for epochs 0–6, 37% at best (epoch 18), case1 / easy 100%. Cause (David): training read an unshuffled `train_loader` — an earlier version of the §4 cell rebuilt both loaders without `shuffle=True`, and the train file is stored in order (positives first), so batches were long runs of one label. Fixed: §4 no longer builds loaders. Notes: the in-kernel check `type(train_loader.sampler).__name__` returned `RandomSampler` after the run; the first (shuffled) run on 2026-10-21 had found 78.7 / val loss 0.024 after epoch 0. The broken run's files were moved out of `checkpoints/`. The §7 checks flag it: only `val loss after epoch 0 = 0.992` fires (no NaN, train loss min 0.0008, found max 37). **Resolved 2026-10-23 (David):** his own sampler check confirms the broken run's loader was sequential (supersedes the `RandomSampler` reading above).
